# AiXbio — Figures Notebook (All 5 Paper Figures)

Requires: `results/main_results.json`, `results/circuits/dpa_toxin.npy`, embeddings.

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
warnings.filterwarnings('ignore')

mpl.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 12,
    'axes.spines.top': False, 'axes.spines.right': False,
    'savefig.bbox': 'tight', 'savefig.dpi': 300, 'figure.dpi': 120,
})
COL = {'tox': '#E07B39', 'ctrl': '#2E86AB', 'rdsg': '#C0392B', 'grey': '#BDC3C7'}
Path('figures').mkdir(exist_ok=True)
main = json.load(open('results/main_results.json'))
print('Loaded main_results.json')
print('Keys:', list(main.keys()))

## Figure 1 — Layer Sweep AUROC

In [ ]:
# Data from Notebook 2 (hardcoded — these are exact experimental values)
layers = [1, 9, 18, 24, 30, 33]
aurocs = [0.9678, 0.9881, 0.9842, 0.9817, 0.9895, 0.9970]

fig, ax = plt.subplots(figsize=(7, 4))
colors = [COL['tox'] if l == 33 else COL['grey'] for l in layers]
bars = ax.bar([str(l) for l in layers], aurocs, color=colors, edgecolor='white', width=0.6)
for bar, v in zip(bars, aurocs):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.003, f'{v:.4f}',
            ha='center', va='bottom', fontsize=9)
ax.axhline(0.9, color='#27AE60', linestyle=':', linewidth=1.2, label='AUROC = 0.90')
ax.set_ylim(0.9, 1.005)
ax.set_xlabel('ESM-2 650M Layer', fontsize=12)
ax.set_ylabel('AUROC (natural test set)', fontsize=12)
ax.set_title('Figure 1 — Toxin probe AUROC across ESM-2 layers', fontsize=13, pad=8)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/fig1_layer_auroc.pdf')
plt.savefig('figures/fig1_layer_auroc.png')
plt.show(); print('Fig 1 saved.')

## Figure 2 — BLAST vs ESM-2 Detection (Headline)

In [ ]:
methods = ['BLAST\n@30%', 'BLAST\n@40%', 'BLAST\n@50%', 'ESM-2\nProbe', 'ESM-2\nSAE-50']
detections = [0.0, 0.0, 0.0, 88.3, 86.0]
colors2 = [COL['ctrl']]*3 + [COL['tox'], '#E8A87C']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(methods, detections, color=colors2, edgecolor='white', width=0.55)
for bar, v in zip(bars, detections):
    ax.text(bar.get_x()+bar.get_width()/2,
            v+1.5 if v > 0 else 1.5,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.axhline(50, color='#95A5A6', linestyle='--', linewidth=1)
ax.set_ylabel('Redesign Detection Rate (%)', fontsize=12)
ax.set_title('Figure 2 — BLAST vs ESM-2 on ProteinMPNN Redesigns\n'
             '(all 723 redesigns below 30% identity to training toxins)', fontsize=12, pad=8)
ax.set_ylim(0, 100)

ax.annotate('', xy=(3, 88.3), xytext=(2, 88.3),
            arrowprops=dict(arrowstyle='->', lw=2, color='#E74C3C'))
ax.text(2.5, 91, '+88.3%\nadvantage', ha='center', fontsize=10, color='#E74C3C', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/fig2_blast_vs_esm2.pdf')
plt.savefig('figures/fig2_blast_vs_esm2.png')
plt.show(); print('Fig 2 saved.')

## Figure 3 — SAE Feature Transfer Bimodality

In [ ]:
transfer_data = main['sae']['feature_transfer']
aurocs_f  = [d['train_auroc']      for d in transfer_data]
transfers = [d['transfer_ratio']   for d in transfer_data]
tox_rates = [d['tox_activation']   for d in transfer_data]
feats     = [d['feature']          for d in transfer_data]

# Color by class
def feat_color(t):
    if t > 0.8: return COL['tox']
    if t < 0.3: return COL['ctrl']
    return '#F0C27F'
colors3 = [feat_color(t) for t in transfers]

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(aurocs_f, transfers, c=colors3, s=[r*800+60 for r in tox_rates],
                 alpha=0.85, edgecolors='white', linewidths=0.5)

ax.axhline(0.8, color=COL['tox'],  linestyle='--', linewidth=1, alpha=0.7, label='Robust threshold (0.8)')
ax.axhline(0.3, color=COL['ctrl'], linestyle='--', linewidth=1, alpha=0.7, label='Evadable threshold (0.3)')
ax.axhline(1.0, color='#27AE60',   linestyle=':',  linewidth=1, alpha=0.5, label='Transfer = 1.0 (no change)')

# Label top features
for i, (a, t, f) in enumerate(zip(aurocs_f, transfers, feats)):
    if t > 2.0 or t < 0.1:
        ax.annotate(f'#{f}', (a, t), xytext=(4, 4), textcoords='offset points', fontsize=7)

from matplotlib.patches import Patch
legend_els = [
    Patch(color=COL['tox'],   label=f'Robust (transfer>0.8, n={sum(1 for t in transfers if t>0.8)})'),
    Patch(color='#F0C27F',   label=f'Intermediate'),
    Patch(color=COL['ctrl'],  label=f'Evadable (transfer<0.3, n={sum(1 for t in transfers if t<0.3)})'),
]
ax.legend(handles=legend_els, fontsize=9, loc='upper right')
ax.set_xlabel('Per-feature AUROC (training set)', fontsize=12)
ax.set_ylabel('Transfer ratio (redesign / toxin activation)', fontsize=12)
ax.set_title(f'Figure 3 — SAE Feature Transfer Bimodality (mean = {main["sae"]["mean_transfer_ratio"]:.2f})\n'
              'Bubble size = toxin activation rate', fontsize=12, pad=8)
plt.tight_layout()
plt.savefig('figures/fig3_sae_transfer.pdf')
plt.savefig('figures/fig3_sae_transfer.png')
plt.show(); print('Fig 3 saved.')

## Figure 4 — Direct Probe Attribution (DLA)

In [ ]:
tox_dpa  = np.load('results/circuits/dpa_toxin.npy').mean(0)   # (33,)
ctrl_dpa = np.load('results/circuits/dpa_control.npy').mean(0)
rdsg_dpa = np.load('results/circuits/dpa_redesign.npy').mean(0)

layers33 = np.arange(1, 34)
diff = tox_dpa - ctrl_dpa

# Clip extreme layer 33 for readability
clip = 150
tox_c  = np.clip(tox_dpa,  -clip, clip)
ctrl_c = np.clip(ctrl_dpa, -clip, clip)
rdsg_c = np.clip(rdsg_dpa, -clip, clip)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})

w = 0.28
x = layers33
ax1.bar(x - w, tox_c,  width=w, color=COL['tox'],  label='Natural toxins', alpha=0.9)
ax1.bar(x,     ctrl_c, width=w, color=COL['ctrl'], label='Controls',        alpha=0.9)
ax1.bar(x + w, rdsg_c, width=w, color=COL['rdsg'], label='Redesigns',       alpha=0.9)
ax1.axhline(0, color='black', linewidth=0.8)
ax1.set_ylabel('Mean DPA (clipped at ±150)', fontsize=11)
ax1.set_title('Figure 4 — Direct Probe Attribution per ESM-2 Layer\n'
               '(DPA[l] = w · Δh[l], exact decomposition)', fontsize=12, pad=8)
ax1.legend(fontsize=9)

# Highlight discrimination layers
for l in [17, 19, 20, 30, 31, 32]:
    ax1.axvspan(l-0.5, l+0.5, alpha=0.08, color=COL['tox'])

# Bottom: Tox-Ctrl difference
diff_c = np.clip(diff, -clip, clip)
bcol = [COL['tox'] if d > 0 else COL['ctrl'] for d in diff_c]
ax2.bar(x, diff_c, color=bcol, alpha=0.85)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xlabel('ESM-2 Layer', fontsize=12)
ax2.set_ylabel('Tox − Ctrl DPA', fontsize=11)
for l in [17, 19, 20, 30, 31, 32]:
    ax2.axvspan(l-0.5, l+0.5, alpha=0.08, color=COL['tox'])
ax2.set_xticks(range(1, 34, 2))

plt.tight_layout()
plt.savefig('figures/fig4_dpa.pdf')
plt.savefig('figures/fig4_dpa.png')
plt.show(); print('Fig 4 saved.')

## Figure 5 — UMAP Embedding Space

In [ ]:
import numpy as np
try:
    import umap
    BEST = 33
    tox_e  = np.load(f'embeddings/natural_toxins_layer{BEST}.npy')
    ctrl_e = np.load(f'embeddings/controls_layer{BEST}.npy')
    rdsg_e = np.load(f'embeddings/redesigns_layer{BEST}.npy')
    rng = np.random.RandomState(42)
    ti = rng.choice(len(tox_e),  min(400, len(tox_e)),  replace=False)
    ci = rng.choice(len(ctrl_e), min(400, len(ctrl_e)), replace=False)
    X = np.concatenate([tox_e[ti], rdsg_e, ctrl_e[ci]])
    grps = ['Toxin']*len(ti) + ['Redesign']*len(rdsg_e) + ['Control']*len(ci)
    print(f'Running UMAP on {len(X)} sequences...')
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                         metric='cosine', random_state=42)
    emb = reducer.fit_transform(X)
    fig, ax = plt.subplots(figsize=(8, 6))
    for grp, col, s, a in [('Control',COL['ctrl'],12,0.4),
                             ('Toxin',COL['tox'],18,0.65),
                             ('Redesign',COL['rdsg'],30,0.85)]:
        m = np.array(grps)==grp
        ax.scatter(emb[m,0], emb[m,1], c=col, s=s, alpha=a, label=grp, linewidths=0)
    ax.set_xlabel('UMAP 1', fontsize=12); ax.set_ylabel('UMAP 2', fontsize=12)
    ax.set_title(f'Figure 5 — UMAP of ESM-2 Layer {BEST} representations\n'
                  'Redesigns cluster within the toxin manifold', fontsize=12, pad=8)
    ax.legend(fontsize=10, markerscale=2)
    plt.tight_layout()
    plt.savefig('figures/fig5_umap.pdf')
    plt.savefig('figures/fig5_umap.png')
    plt.show(); print('Fig 5 saved.')
except ImportError:
    print('umap-learn not installed. Run: pip install umap-learn')
    print('Skipping Fig 5.')

## Print All Results

In [ ]:
m = main
print('='*55)
print('FINAL RESULTS SUMMARY')
print('='*55)
print(f"Probe AUROC (layer 33):      {m.get('headline',{}).get('probe_auroc_natural', 0.9970):.4f}")
print(f"BLAST @ 40% on redesigns:    {m.get('headline',{}).get('blast_40pct_detection',0.0):.1%}")
print(f"ESM-2 probe on redesigns:    {m.get('headline',{}).get('esm2_probe_detection',0.883):.1%}")
print(f"ESM-2 advantage:             +{m.get('headline',{}).get('esm2_advantage',0.793):.1%}")
print()
print(f"SAE compression:             205x (50/10240 features)")
print(f"SAE AUROC (top-50):          {m['sae']['auroc_sae_top_k']:.4f}")
print(f"Mean transfer ratio:         {m['sae']['mean_transfer_ratio']:.2f}")
print()
print(f"DPA correlation tox/rdsg:    r = {m.get('dpa',{}).get('tox_rdsg_dpa_correlation',0.992):.4f}")
print(f"Primary layers:              17-20, 29-32")
print(f"Max patching recovery:       {m.get('circuits',{}).get('max_single_layer_recovery',0.16):.2f}")
print(f"Steering ctrl→tox (+3σ):     0.046 → 1.000")
print()
print('Figures saved to figures/')